<a href="https://colab.research.google.com/github/jackleard/ml-text-classifier/blob/main/ml-text-classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install planetterp pandas torch transformers scikit-learn

In [ ]:
# 1. Data collection (PlanetTerp API)

import planetterp
import pandas as pd

PROFESSORS = [
    "Fardina Alam",
    "Nelson Padua-Perez",
    "Cliff Bakalian",
    "Pedram Sadeghian",
    "Fawzi Emad",
]

def fetch_reviews_for_prof(name: str):
    # Get professor data + their reviews
    prof_data = planetterp.professor(name=name, reviews=True)
    reviews = prof_data.get("reviews", []) or []

    cleaned = []
    for r in reviews:
        cleaned.append({
            "professor": prof_data.get("name", name),
            "course": r.get("course"),
            "stars": r.get("rating"),       # 1–5
            "review_text": r.get("review"), # main text
            "created": r.get("created"),
        })
    return cleaned

all_reviews = []
for prof in PROFESSORS:
    print(f"Fetching {prof}...")
    all_reviews.extend(fetch_reviews_for_prof(prof))

df = pd.DataFrame(all_reviews)
print(df.head())
print("Total reviews:", len(df))

df.to_csv("professor_reviews.csv", index=False)



In [ ]:
# 2. Data loading and preprocessing
import pandas as pd

df = pd.read_csv("professor_reviews.csv")

# Drop missing or empty reviews / ratings
df = df.dropna(subset=["review_text", "stars"])
df["review_text"] = df["review_text"].astype(str).str.strip()
df = df[df["review_text"] != ""]

# Ratings should be integers 1–5
df["stars"] = df["stars"].astype(int)
df = df[(df["stars"] >= 1) & (df["stars"] <= 5)]

print(df["stars"].value_counts())
print("Final dataset size:", len(df))

from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["stars"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["stars"],
    random_state=42
)

len(train_df), len(val_df), len(test_df)

In [ ]:
# Sample a few example reviews for the slide
df[["professor", "course", "stars", "review_text"]].sample(5, random_state=42)

# Rating distribution and split sizes for slide
import matplotlib.pyplot as plt

star_counts = df["stars"].value_counts().sort_index()

plt.figure(figsize=(5, 4))
plt.bar(star_counts.index.astype(str), star_counts.values)
plt.xlabel("Star rating")
plt.ylabel("Number of reviews")
plt.title("Distribution of Star Ratings (1–5)")
plt.tight_layout()
plt.show()

sizes = {
    "Split": ["Train", "Validation", "Test"],
    "Count": [len(train_df), len(val_df), len(test_df)]
}

split_df = pd.DataFrame(sizes)
split_df

In [ ]:
# 3. Model and tokenizer

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

MODEL_NAME = "distilbert-base-uncased"  # good small model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

num_labels = 5  # for 1–5 stars
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)

model.to(device)

learning_rate = 2e-5
optimizer = AdamW(model.parameters(), lr=learning_rate)

In [ ]:
# 4. Dataset and dataloaders

class ReviewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.texts = df["review_text"].tolist()
        # Map stars 1–5 → labels 0–4
        self.labels = (df["stars"].values - 1).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }
        return item

train_dataset = ReviewsDataset(train_df, tokenizer)
val_dataset   = ReviewsDataset(val_df, tokenizer)
test_dataset  = ReviewsDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16)
test_loader  = DataLoader(test_dataset, batch_size=16)

In [ ]:
# 5. Training

epochs = 3
learning_rate = 2e-5

optimizer = AdamW(model.parameters(), lr=learning_rate)

total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

from tqdm.auto import tqdm
import numpy as np

def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0

    for batch in tqdm(loader, desc="Training"):
        optimizer.zero_grad()

        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    return avg_loss

def eval_model(model, loader):
    model.eval()
    total_loss = 0
    preds = []
    labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Validation"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"]
            )
            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())

    avg_loss = total_loss / len(loader)
    return avg_loss, np.array(preds), np.array(labels)

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler)
    val_loss, val_preds, val_labels = eval_model(model, val_loader)

    val_acc = (val_preds == val_labels).mean()
    print(f"Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | Val acc: {val_acc:.4f}")




In [ ]:
# 6. Evaluation and examples

test_loss, test_preds, test_labels = eval_model(model, test_loader)
test_acc = (test_preds == test_labels).mean()
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}")

test_true_stars = test_labels + 1
test_pred_stars = test_preds + 1

from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(test_true_stars, test_pred_stars, digits=3))

cm = confusion_matrix(test_true_stars, test_pred_stars)
cm

test_df = test_df.reset_index(drop=True)

for i in range(5):
    text = test_df.loc[i, "review_text"]
    true_star = test_df.loc[i, "stars"]

    enc = tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**enc)
        pred_label = torch.argmax(outputs.logits, dim=1).item()

    pred_star = pred_label + 1

    print("="*80)
    print(text[:500], "...")
    print(f"True stars: {true_star} | Pred stars: {pred_star}")

In [ ]:
# Visuals for Slides 6 and 7

import matplotlib.pyplot as plt

# Majority-class baseline: always predict the most common star rating
majority_class_freq = df["stars"].value_counts(normalize=True).max()
print("Majority class baseline accuracy:", majority_class_freq)

random_baseline = 1.0 / 5.0   # 5 classes
model_acc = test_acc          # from eval_model above

labels = ["Random", "Majority class", "Our model"]
values = [random_baseline, majority_class_freq, model_acc]

plt.figure(figsize=(5, 4))
plt.bar(labels, values)
plt.ylim(0, 1.0)
plt.ylabel("Accuracy")
plt.title("Accuracy: Baselines vs Our Model")

for i, v in enumerate(values):
    plt.text(i, v + 0.02, f"{v:.2f}", ha="center")

plt.tight_layout()
plt.show()

import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm)

# Tick labels
ax.set_xticks(np.arange(5))
ax.set_yticks(np.arange(5))
ax.set_xticklabels([1, 2, 3, 4, 5])
ax.set_yticklabels([1, 2, 3, 4, 5])
ax.set_xlabel("Predicted rating")
ax.set_ylabel("True rating")
ax.set_title("Confusion Matrix (Test Set)")

# Rotate x tick labels
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Annotate counts
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8)

fig.tight_layout()
plt.show()